# IndoLens — Pipeline Training & Analisis Dataset Skripsi
Notebook ini dirancang untuk memproses dataset wajah aktor Indonesia secara offline menggunakan akselerasi CPU/GPU dan memuat reference embedding ke runtime IndoLens.

**Urutan Eksekusi:**
1. **Cell 0**: Setup & Pre-Flight Check (verifikasi package, MySQL read-only, dan path file)
2. **Stage 01**: Ekstraksi Wajah (YOLOv8-Face)
3. **Stage 02**: Analisis Matematis Dataset & Perceptual Hashing (dHash)
4. **Stage 03**: Preprocessing Wajah (160x160, Normalisasi VGGFace2)
5. **Stage 04**: Pembuatan Embedding 128-D (FaceNet Inception-ResNet-v1)
6. **Stage 05**: Validasi Embedding & Audit Jarak Euclidean (Intra-class vs Inter-class)
7. **Stage 06**: Integrasi Reference Data ke Runtime Web (`python/embeddings/`)
8. **Stage 07**: Evaluasi Akhir Dataset & Rekapitulasi Metrik Skripsi

In [ ]:
# CELL 0: SETUP & PRE-FLIGHT CHECK
import os, sys
from pathlib import Path

_ROOT = Path(os.environ.get('INDOLENS_ROOT', r'C:\laragon\www\IndoLens'))
for _p in [_ROOT, _ROOT / 'training_python', _ROOT / 'python']:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import torch
import training_python as tp
from training_python import tp_config as cfg
from training_python import tp_database as db

print('[OK] Project Root:', cfg.PROJECT_ROOT)
print('[OK] PyTorch     :', torch.__version__)
print('[OK] GPU / CUDA  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

records = db.fetch_actor_image_records()
print(f'[OK] MySQL Connected (Read-Only): Ditemukan {len(records)} record gambar aktor terdaftar.')

## Stage 01: Ekstraksi Wajah (YOLOv8-Face)
- Membaca daftar gambar resmi dari MySQL (Read-Only).
- Mendeteksi bounding box wajah menggunakan YOLOv8-Face.
- Menyimpan hasil crop wajah asli ke `artifacts/01_ekstraksi_wajah/crops/`.
- Menangani multi-face (disimpan ke `rejected/` untuk audit) dan zero-face.

In [ ]:
# Tips: Anda bisa menguji 1 gambar pertama dengan: tp.run_stage_01(limit=1)
# Jalankan untuk memproses seluruh gambar (aman diinterupsi / resume otomatis):
tp.run_stage_01()

## Stage 02: Analisis Matematis Dataset
- Menghitung statistik dimensi gambar asli & crop wajah (min, max, mean, std).
- Menghitung distribusi skor confidence deteksi YOLO.
- Mendeteksi duplikasi visual dengan perceptual difference hashing (dHash 64-bit).

In [ ]:
tp.run_stage_02()

## Stage 03: Preprocessing Wajah
- Konversi BGR -> RGB.
- Resize presisi ke 160x160 piksel menggunakan bilinear interpolation.
- Normalisasi float32 VGGFace2 standardization: `(x - mean) / std`.
- Menyimpan tensor ke `artifacts/03_preprocessing_wajah/tensors/`.

In [ ]:
tp.run_stage_03()

## Stage 04: Pembuatan Embedding 128-D
- Inferensi model neural network FaceNet Inception-ResNet-v1 (pretrained VGGFace2).
- Truncate ke 128 dimensi & normalisasi Euclidean L2-Norm (unit length = 1.0).
- Menyimpan vektor embedding `.npy` ke `artifacts/04_pembuatan_embedding/vectors/`.

In [ ]:
tp.run_stage_04()

## Stage 05: Validasi Kualitas Embedding & Audit Jarak Euclidean
- Validasi bebas NaN, Inf, dimensi tepat 128, dan L2-norm == 1.0.
- Menghitung matriks jarak Euclidean intra-class (foto aktor yang sama).
- Menandai pasangan foto dengan variasi sudut/pencahayaan ekstrem (anomali).

In [ ]:
tp.run_stage_05()

## Stage 06: Integrasi Reference Data ke Runtime Web
- Membuat staging data terisolasi & backup otomatis direktori `python/embeddings/`.
- Mempromosikan vektor reference data yang lolos validasi ke folder runtime `python/embeddings/<actor_slug>/`.
- Mengirim sinyal reload memori ke service IndoLens (jika service sedang aktif).

In [ ]:
tp.run_stage_06(promote_to_production=True)

## Stage 07: Evaluasi Akhir Dataset & Rekapitulasi Metrik Skripsi
- Rekapitulasi seluruh metrik penelitian (deteksi YOLO, distribusi crop, audit jarak).
- Evaluasi opsional terhadap folder `probe_set/` jika disediakan foto uji independen.
- Menghasilkan laporan resmi `evaluation_report.json` dan `evaluation_report.txt`.

In [ ]:
tp.run_stage_07()